#Instalacja
Biblioteka ucimlrepo - umożliwia bezposrednie pobieranie zbiorow danych z repozytorium UCI Machine Learning Repository do Python

In [ ]:
!pip install ucimlrepo

#Pobieranie danych, czyszczenie klas


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np


# pobranie datasetu z UCI
dataset = fetch_ucirepo(id=257)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

# ujednolicenie etykiet klas do dokładnie 4 (very_low i Very Low na Very Low)
# 1) usuwam spacje, 2) ujednolicam wielkość liter, 3) ujednolicam zapis "Very Low"
df["UNS"] = df["UNS"].astype(str).str.strip().str.lower()
df["UNS"] = df["UNS"].replace({"very low": "very_low"})

# Nazwy do raportu duzych liter
df["UNS"] = df["UNS"].map({
    "low": "Low",
    "middle": "Middle",
    "high": "High",
    "very_low": "Very Low"
})

print("Liczba klas:", df["UNS"].nunique())
print(df["UNS"].value_counts())
df.head()

Liczba klas: 4
UNS
Low         129
Middle      122
High        102
Very Low     50
Name: count, dtype: int64


,STG,SCG,STR,LPR,PEG,UNS
0,0.00,0.00,0.00,0.00,0.00,Very Low
1,0.08,0.08,0.10,0.24,0.90,High
2,0.06,0.06,0.05,0.25,0.33,Low
3,0.10,0.10,0.15,0.65,0.30,Middle
4,0.08,0.08,0.08,0.98,0.24,Low


#Pełna charakterystyka danych

In [ ]:
print("Wymiary zbioru (wiersze, kolumny):", df.shape)

print("\nKolumny:")
print(list(df.columns))

print("\nTypy danych:")
print(df.dtypes)

print("\nBraki danych (na kolumnę):")
print(df.isnull().sum())

print("\nStatystyki opisowe cech numerycznych:")
print(df.drop(columns=["UNS"]).describe().T)

print("\nRozkład klas (UNS):")
print(df["UNS"].value_counts())

Wymiary zbioru (wiersze, kolumny): (403, 6)

Kolumny:
['STG', 'SCG', 'STR', 'LPR', 'PEG', 'UNS']

Typy danych:
STG    float64
SCG    float64
STR    float64
LPR    float64
PEG    float64
UNS     object
dtype: object

Braki danych (na kolumnę):
STG    0
SCG    0
STR    0
LPR    0
PEG    0
UNS    0
dtype: int64

Statystyki opisowe cech numerycznych:
     count      mean       std  min    25%   50%   75%   max
STG  403.0  0.353141  0.212018  0.0  0.200  0.30  0.48  0.99
SCG  403.0  0.355940  0.215531  0.0  0.200  0.30  0.51  0.90
STR  403.0  0.457655  0.246684  0.0  0.265  0.44  0.68  0.95
LPR  403.0  0.431342  0.257545  0.0  0.250  0.33  0.65  0.99
PEG  403.0  0.456360  0.266775  0.0  0.250  0.40  0.66  0.99

Rozkład klas (UNS):
UNS
Low         129
Middle      122
High        102
Very Low     50
Name: count, dtype: int64


#Sprawdzenie spójności decyzyjnej

In [ ]:
feature_cols = ["STG", "SCG", "STR", "LPR", "PEG"]

inconsistent = (
    df.groupby(feature_cols)["UNS"]
    .nunique()
    .reset_index(name="class_count")
)

inconsistent_cases = inconsistent[inconsistent["class_count"] > 1]

print("Liczba niespójnych przypadków:", len(inconsistent_cases))

Liczba niespójnych przypadków: 0


#Duplikaty i korelacje

In [ ]:
# duplikaty pełne (identyczne wiersze)
dup_full = df.duplicated().sum()
print("Duplikaty pełne:", int(dup_full))

print("\nKorelacje między atrybutami:")
print(df.corr(numeric_only=True))

Duplikaty pełne: 0

Korelacje między atrybutami:
          STG       SCG       STR       LPR       PEG
STG  1.000000  0.049023 -0.051889  0.113957  0.198629
SCG  0.049023  1.000000  0.121235  0.119716  0.193566
STR -0.051889  0.121235  1.000000  0.083423  0.148338
LPR  0.113957  0.119716  0.083423  1.000000 -0.039283
PEG  0.198629  0.193566  0.148338 -0.039283  1.000000
